# Exercise 1: SAXPY: solution

Solution to Exercise 1 in `gpu_introduction.ipynb`.

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

  Activating project at `c:\Users\evely\OneDrive\Documents\CSE_MIT\julia-hpc-tutorial-sc26\02-Parallel-Programming\KernelAbstractions`


In [2]:
using KernelAbstractions

# Pick your hardware: :NVIDIA, :AMD, :Metal, or :CPU
DEV = :NVIDIA

if DEV == :NVIDIA
    using CUDA
    ArrayKA = CUDA.CuArray
    Backend = CUDA.CUDABackend()
elseif DEV == :AMD
    using AMDGPU
    ArrayKA = AMDGPU.ROCArray
    Backend = AMDGPU.ROCBackend()
elseif DEV == :Metal
    using Metal
    ArrayKA = Metal.MtlArray
    Backend = Metal.MetalBackend()
elseif DEV == :CPU
    ArrayKA = Array
    Backend = CPU()
else
    error("Unknown DEV = $DEV. Use :NVIDIA, :AMD, :Metal, or :CPU.")
end

CUDABackend(false, false, false)

In [3]:
@kernel function my_saxpy_gpu_kernel!(y, α, x)
    i = @index(Local)
    y[i] = α * x[i] + y[i]
end

my_saxpy_gpu_kernel! (generic function with 4 methods)

In [4]:
α = 2.0f0
x = rand(Float32, 1024)
y = rand(Float32, 1024)

dx = allocate(Backend, Float32, length(x))
copyto!(dx, x)
dy = allocate(Backend, Float32, length(y))
copyto!(dy, y)

saxpy! = my_saxpy_gpu_kernel!(Backend, length(dx))
saxpy!(dy, α, dx, ndrange=length(dx))
KernelAbstractions.synchronize(Backend)

Array(dy) ≈ α .* x .+ y

true